# 🔨 Crack 3D Reconstruction + Analysis
Run cells 1-5 in order. **GPU runtime recommended** (Runtime → Change runtime type → T4 GPU).

Each run produces:
- **3D Crack Mesh tab** — interactive GLB viewer with vertex colors
- **Analysis Report tab** — 6-panel figure with severity classification and stats

In [ ]:
# ── Cell 1: Install ───────────────────────────────────────────────────────────
# Run once. If Colab asks to restart runtime after install, do so, then re-run
# from Cell 2 onward (no need to re-install).
%pip install -q gradio open3d transformers pillow accelerate opencv-python matplotlib trimesh pygltflib


In [ ]:
# ── Cell 2: Imports & model ───────────────────────────────────────────────────
import os, tempfile, traceback
import cv2
import numpy as np
from PIL import Image
from scipy.spatial import cKDTree

import matplotlib
matplotlib.use("Agg")          # must be set BEFORE importing pyplot
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap

import torch
from transformers import pipeline

import open3d as o3d
import trimesh                 # reliable GLB export with vertex colors
import gradio as gr

# ── Device ────────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    device = 0
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
else:
    device = -1
    print("⚠️  CPU only — inference will be slow")

# ── Depth model ───────────────────────────────────────────────────────────────
print("Loading Depth-Anything-V2-Large  (~1.3 GB, cached after first run)…")
depth_pipe = pipeline(
    "depth-estimation",
    model="depth-anything/Depth-Anything-V2-Large-hf",
    device=device,
)
print("✅ Model ready.")


In [ ]:
# ── Cell 3: Helpers ───────────────────────────────────────────────────────────

# ── 3a. Severity classifier ───────────────────────────────────────────────────
def classify_severity(max_disp: float, crack_area_pct: float):
    """Return (label, hex_color).  max_disp is in the 0-1000 normalised scale."""
    score = (max_disp / 1000.0) * 0.6 + (crack_area_pct / 100.0) * 0.4
    if score < 0.05:   return "Hairline", "#4CAF50"
    if score < 0.15:   return "Minor",    "#8BC34A"
    if score < 0.30:   return "Moderate", "#FFC107"
    if score < 0.50:   return "Severe",   "#FF5722"
    return             "Critical",        "#B71C1C"


# ── 3b. Figure-to-numpy (matplotlib >= 3.8 compatible) ───────────────────────
def fig_to_numpy(fig) -> np.ndarray:
    """Render a matplotlib figure to an HxWx3 uint8 RGB array."""
    fig.canvas.draw()
    buf = fig.canvas.buffer_rgba()          # returns memoryview, works on all versions
    w, h = fig.canvas.get_width_height()
    arr = np.frombuffer(buf, dtype=np.uint8).reshape(h, w, 4)
    return arr[:, :, :3].copy()             # drop alpha → RGB


# ── 3c. 6-panel report builder ────────────────────────────────────────────────
def build_report(original_rgb, base_depth, crack_mask_raw,
                 displacement_map, final_depth, crack_intensity) -> np.ndarray:
    """
    Returns HxWx3 uint8 array.

    Panels:
      [0,0] Original image
      [0,1] AI base depth map          (inferno)
      [0,2] Binary crack mask          (gray)
      [1,0] Displacement heatmap       (jet — crack carving depth)
      [1,1] Depth difference Δ         (plasma — final minus base)
      [1,2] Severity classification overlay
    """
    crack_px       = (crack_mask_raw > 10).astype(np.uint8)
    crack_area_pct = float(crack_px.mean() * 100.0)
    max_disp       = float(displacement_map.max())
    mean_disp      = (float(displacement_map[crack_px == 1].mean())
                      if crack_px.any() else 0.0)
    depth_diff     = (final_depth - base_depth).astype(np.float32)
    sev_label, _   = classify_severity(max_disp, crack_area_pct)

    # Severity color overlay
    sev_cmap  = LinearSegmentedColormap.from_list(
        "sev", ["#4CAF50", "#FFC107", "#FF5722", "#B71C1C"])
    disp_norm = displacement_map / (max_disp + 1e-6)
    sev_rgb   = (sev_cmap(disp_norm)[:, :, :3] * 255).astype(np.uint8)
    alpha     = (crack_px[:, :, None] * 0.80).astype(np.float32)
    bg        = (original_rgb * 0.30).astype(np.uint8)
    sev_over  = np.clip(sev_rgb * alpha + bg * (1 - alpha), 0, 255).astype(np.uint8)

    # Figure
    fig, axes = plt.subplots(2, 3, figsize=(18, 12),
                             facecolor="#1a1a1a",
                             gridspec_kw={"hspace": 0.35, "wspace": 0.10})
    fig.patch.set_facecolor("#1a1a1a")

    panels = [
        (original_rgb,     "Original Image",          None,       False),
        (base_depth,       "AI Base Depth Map",        "inferno",  True),
        (crack_mask_raw,   "Crack Binary Mask",        "gray",     True),
        (displacement_map, "Displacement Heatmap",     "jet",      True),
        (depth_diff,       "Depth Difference Δ",       "plasma",   True),
        (sev_over,         "Severity Classification",  None,       False),
    ]

    for ax, (data, title, cmap, cbar) in zip(axes.flat, panels):
        ax.set_facecolor("#1a1a1a")
        im = ax.imshow(data, cmap=cmap, interpolation="bilinear")
        if cbar:
            cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            cb.ax.yaxis.set_tick_params(color="#aaa", labelsize=7)
            plt.setp(cb.ax.yaxis.get_ticklabels(), color="#aaa")
            cb.outline.set_edgecolor("#555")
        ax.set_title(title, color="#eee", fontsize=12, fontweight="bold", pad=7)
        ax.axis("off")

    # Stats under severity panel
    axes[1, 2].text(
        0.5, -0.07,
        (f"Severity: {sev_label}   Max Δ: {max_disp:.1f}   "
         f"Mean Δ: {mean_disp:.1f}   Crack area: {crack_area_pct:.2f}%"),
        transform=axes[1, 2].transAxes,
        ha="center", va="top", fontsize=9, color="#ddd",
        bbox=dict(boxstyle="round,pad=0.4",
                  facecolor="#2a2a2a", edgecolor="#555", linewidth=0.8),
    )

    # Legend
    patches = [
        mpatches.Patch(color="#4CAF50", label="Hairline"),
        mpatches.Patch(color="#8BC34A", label="Minor"),
        mpatches.Patch(color="#FFC107", label="Moderate"),
        mpatches.Patch(color="#FF5722", label="Severe"),
        mpatches.Patch(color="#B71C1C", label="Critical"),
    ]
    axes[1, 2].legend(handles=patches, loc="upper right", fontsize=7,
                      framealpha=0.55, facecolor="#1a1a1a",
                      edgecolor="#555", labelcolor="#eee")

    fig.suptitle(
        f"Crack Analysis  ·  Displacement strength: {crack_intensity}"
        f"  ·  Overall severity: {sev_label}",
        color="#fff", fontsize=14, fontweight="bold", y=0.99)

    arr = fig_to_numpy(fig)
    plt.close(fig)
    return arr


print("✅ Helpers ready.")


In [ ]:
# ── Cell 4: Core reconstruction ───────────────────────────────────────────────

def _to_glb_with_colors(mesh_o3d, out_path: str) -> str:
    """
    Export an Open3D TriangleMesh WITH vertex colors to GLB using trimesh.
    open3d's own write_triangle_mesh drops vertex colors in most GLB builds.
    """
    verts  = np.asarray(mesh_o3d.vertices)
    tris   = np.asarray(mesh_o3d.triangles)
    colors = np.asarray(mesh_o3d.vertex_colors)          # float [0,1]
    colors_u8 = (np.clip(colors, 0, 1) * 255).astype(np.uint8)
    # trimesh wants RGBA vertex colors
    rgba = np.hstack([colors_u8,
                      np.full((len(colors_u8), 1), 255, dtype=np.uint8)])
    tm = trimesh.Trimesh(vertices=verts, faces=tris,
                         vertex_colors=rgba, process=False)
    tm.export(out_path)
    return out_path


def reconstruct_crack(input_image, crack_intensity: float = 50):
    """
    Returns (glb_path_or_None, report_PIL_Image_or_None).
    Both outputs may be independently None if the respective step fails.
    """
    report_pil = None     # initialise so the except branch can always return it

    if input_image is None:
        return None, None

    try:
        # ── 0. Channel normalisation ───────────────────────────────────────
        img = input_image
        if img.ndim == 2:
            img = np.stack([img] * 3, axis=-1)
        elif img.shape[2] == 4:
            img = img[:, :, :3]
        img = img.astype(np.uint8)

        # ── 1. Crack mask → displacement ──────────────────────────────────
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        _, mask_raw = cv2.threshold(gray, 120, 255, cv2.THRESH_BINARY_INV)
        mask_blur   = cv2.GaussianBlur(mask_raw, (15, 15), 0).astype(np.float32)
        disp_map    = (mask_blur / 255.0) * float(crack_intensity)

        # ── 2. Depth-Anything inference ────────────────────────────────────
        pil_in     = Image.fromarray(img)
        base_depth = np.array(depth_pipe(pil_in)["depth"]).astype(np.float32)

        # ── 3. Align sizes ─────────────────────────────────────────────────
        dh, dw = base_depth.shape
        if (img.shape[0], img.shape[1]) != (dh, dw):
            disp_map = cv2.resize(disp_map,  (dw, dh), interpolation=cv2.INTER_LINEAR)
            mask_raw = cv2.resize(mask_raw,  (dw, dh), interpolation=cv2.INTER_NEAREST)
            img      = cv2.resize(img,       (dw, dh), interpolation=cv2.INTER_LINEAR)

        # ── 4. Combine depth + crack carving ──────────────────────────────
        final_depth = base_depth + disp_map
        d_min, d_max = final_depth.min(), final_depth.max()
        depth_norm   = (final_depth - d_min) / (d_max - d_min + 1e-6)
        depth_uint   = (depth_norm * 1000).astype(np.uint16)

        # ── 5. Analysis report (built BEFORE slow Poisson step) ───────────
        report_arr = build_report(
            original_rgb    = img,
            base_depth      = base_depth,
            crack_mask_raw  = mask_raw,
            displacement_map = disp_map,
            final_depth     = final_depth,
            crack_intensity = crack_intensity,
        )
        report_pil = Image.fromarray(report_arr)   # Gradio Image output needs PIL
        print("✅ Report rendered.")

        # ── 6. RGBD point cloud ────────────────────────────────────────────
        color_o3d = o3d.geometry.Image(np.ascontiguousarray(img))
        depth_o3d = o3d.geometry.Image(np.ascontiguousarray(depth_uint))
        rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(
            color_o3d, depth_o3d,
            depth_scale=1000.0, depth_trunc=3.0,
            convert_rgb_to_intensity=False)

        h, w  = depth_uint.shape
        fx = fy = float(max(h, w))
        intr  = o3d.camera.PinholeCameraIntrinsic(w, h, fx, fy, w/2.0, h/2.0)
        pcd   = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd, intr)
        pcd.transform([[1,0,0,0],[0,-1,0,0],[0,0,-1,0],[0,0,0,1]])

        # ── 7. Downsample + clean ──────────────────────────────────────────
        pcd = pcd.voxel_down_sample(voxel_size=0.005)
        n   = len(pcd.points)
        if n < 500:
            print(f"⚠️  Only {n} points after downsample — skipping 3D mesh.")
            return None, report_pil
        pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
        pcd.estimate_normals(
            search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.02, max_nn=30))
        pcd.orient_normals_towards_camera_location([0.0, 0.0, 0.0])

        # ── 8. Poisson mesh ────────────────────────────────────────────────
        mesh, _ = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
            pcd, depth=8)
        mesh = mesh.crop(pcd.get_axis_aligned_bounding_box())

        # ── 9. Vertex color bake (vectorised) ─────────────────────────────
        pc_pts  = np.asarray(pcd.points)
        pc_col  = np.asarray(pcd.colors)
        mv      = np.asarray(mesh.vertices)
        _, idx  = cKDTree(pc_pts).query(mv)          # no workers arg = portable
        mesh.vertex_colors = o3d.utility.Vector3dVector(pc_col[idx])

        # ── 10. Export GLB via trimesh (vertex colors preserved) ───────────
        glb_path = os.path.join(tempfile.gettempdir(), "crack_mesh.glb")
        _to_glb_with_colors(mesh, glb_path)
        print(f"✅ Mesh → {glb_path}  ({len(mesh.triangles):,} tris)")

        return glb_path, report_pil

    except Exception:
        traceback.print_exc()
        return None, report_pil    # still return report if we have it


print("✅ reconstruct_crack() defined.")


In [ ]:
# ── Cell 5: Gradio UI ─────────────────────────────────────────────────────────

with gr.Blocks(title="Crack 3D Reconstruction", theme=gr.themes.Soft()) as demo:

    gr.Markdown(
        "## 🪨 Crack 3D Reconstruction + Analysis\n"
        "Upload a crack photo → get an **interactive 3D mesh** and a "
        "**6-panel analysis report** with severity classification."
    )

    with gr.Row():
        with gr.Column(scale=1, min_width=320):
            input_img = gr.Image(type="numpy", label="📷 Crack Image")
            intensity = gr.Slider(
                minimum=10, maximum=150, value=50, step=5,
                label="Crack Displacement Strength",
                info="10–40 = hairline  |  50–80 = moderate  |  90–150 = deep structural")
            run_btn = gr.Button("⚙️ Generate 3D Model + Report", variant="primary")
            gr.Markdown(
                "_Tips: use a well-lit, high-contrast photo. "
                "The report is generated first and will appear even if the mesh step fails._"
            )

    with gr.Tabs():
        with gr.Tab("🧊 3D Crack Mesh"):
            output_model = gr.Model3D(
                label="Interactive 3D Mesh",
                clear_color=[0.12, 0.12, 0.12, 1.0],
            )
            gr.Markdown("_Drag to rotate · Scroll to zoom · Right-drag to pan_")

        with gr.Tab("📊 Analysis Report"):
            output_report = gr.Image(
                label="6-Panel Crack Analysis",
                show_download_button=True,
                # NO type= arg here — output Image accepts PIL by default
            )
            gr.Markdown(
                "**Panels:** Original · AI depth · Crack mask · "
                "Displacement heatmap · Depth difference Δ · Severity overlay\n\n"
                "Stats (max Δ, mean Δ, crack area %) appear beneath the severity panel."
            )

    run_btn.click(
        fn=reconstruct_crack,
        inputs=[input_img, intensity],
        outputs=[output_model, output_report],
    )

demo.launch(share=True)
